#개요
Olist 데이터셋 가지고 text-to-sql FineTuning용 데이터셋 만들기.

### 전체 flow

1. 환경 설정
- df 이름, table 이름

2. LLM 실행
- DDL Statement
- 칼럼 설명
- 칼럼별 unique한 값 예시
- base dataset 질문 예시 (실제 질문처럼 하기 위해)

3. 응답 파싱

4. 질문 말투 다양화
- 칼럼명 간접 언급, 명사구 질문, 종결 어미 변경

5. 결과 저장
- DDL문 뒤 INSERT INTO - VALUES - 랜덤 개수 추가



#환경설정

In [2]:
!pip install langchain-openai langchain

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 119.8/119.8 kB 2.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 557.4/557.4 kB 9.1 MB/s eta 0:00:00
  Attempting uninstall: langchain-protocol
    Found existing installation: langchain-protocol 0.0.16
    Uninstalling langchain-protocol-0.0.16:
      Successfully uninstalled langchain-protocol-0.0.16
  Attempting uninstall: langchain-core
    Found existing installation: langchain-core 1.4.3
    Uninstalling langchain-core-1.4.3:
      Successfully uninstalled langchain-core-1.4.3


In [3]:
import json
import re
import pandas as pd
import random
import sqlite3
import os
from langchain_openai import ChatOpenAI
from langchain_core.prompts import ChatPromptTemplate, SystemMessagePromptTemplate, HumanMessagePromptTemplate

In [ ]:
os.environ["OPENAI_API_KEY"] = ""

In [115]:
llm = ChatOpenAI(model = "gpt-5", temperature = 0.3)
llm_sql = ChatOpenAI(model = 'gpt-4o-mini', temperature=0)
llm_small = ChatOpenAI(model = 'gpt-4o-mini', temperature = 0.7)

목표

instruction : "DDL Statements:DDL문\n입력:한글프롬프트"

input : ""

output : SQL 문

In [6]:
with open('/content/gretelai_text_to_sql_data.json', 'r', encoding='utf-8') as f:
    data = json.load(f)

print(json.dumps(data, indent=2, ensure_ascii=False))

[
  {
    "instruction": "DDL statements:\nCREATE TABLE salesperson (salesperson_id INT, name TEXT, region TEXT); INSERT INTO salesperson (salesperson_id, name, region) VALUES (1, 'John Doe', 'North'), (2, 'Jane Smith', 'South'); CREATE TABLE timber_sales (sales_id INT, salesperson_id INT, volume REAL, sale_date DATE); INSERT INTO timber_sales (sales_id, salesperson_id, volume, sale_date) VALUES (1, 1, 120, '2021-01-01'), (2, 1, 150, '2021-02-01'), (3, 2, 180, '2021-01-01');\n입력 텍스트: 각 판매원이 판매한 목재의 총량은 얼마이며, 판매원에 따라 정렬되어 있나요?\n\n위의 테이블 명세와 사용자의 입력 텍스트를 바탕으로 SQL 쿼리를 작성합니다.",
    "input": "",
    "output": "쿼리 작성: SELECT salesperson_id, name, SUM(volume) as total_volume FROM timber_sales JOIN salesperson ON timber_sales.salesperson_id = salesperson.salesperson_id GROUP BY salesperson_id, name ORDER BY total_volume DESC;"
  },
  {
    "instruction": "DDL statements:\nCREATE TABLE equipment_maintenance (equipment_type VARCHAR(255), maintenance_frequency INT);\n입력 텍스트: 장비 유형과 해당 장비의 전체 유지 보

데이터셋 로드

In [188]:
#Olist 데이터
df_customers        = pd.read_csv('olist_customers_dataset.csv', dtype={'customer_zip_code_prefix': str})
df_geolocation      = pd.read_csv('olist_geolocation_dataset.csv', dtype={'geolocation_zip_code_prefix': str})
df_order_items      = pd.read_csv('olist_order_items_dataset.csv')
df_order_payments   = pd.read_csv('olist_order_payments_dataset.csv')
df_order_reviews    = pd.read_csv('olist_order_reviews_dataset.csv')
df_orders           = pd.read_csv('olist_orders_dataset.csv')
df_products         = pd.read_csv('olist_products_dataset.csv')
df_sellers          = pd.read_csv('olist_sellers_dataset.csv', dtype={'seller_zip_code_prefix': str})

In [8]:
#base text-to-sql 데이터
!wget https://raw.githubusercontent.com/leejunho12316/LLaMA-Factory/main/data/text_to_sql_data.json

--2026-06-20 03:12:30--  https://raw.githubusercontent.com/leejunho12316/LLaMA-Factory/main/data/text_to_sql_data.json
Resolving raw.githubusercontent.com (raw.githubusercontent.com)... 185.199.111.133, 185.199.109.133, 185.199.108.133, ...
Connecting to raw.githubusercontent.com (raw.githubusercontent.com)|185.199.111.133|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 3676178 (3.5M) [application/octet-stream]
Saving to: ‘text_to_sql_data.json’

text_to_sql_data.js 100%[===================>]   3.51M  20.9MB/s    in 0.2s    

2026-06-20 03:12:31 (20.9 MB/s) - ‘text_to_sql_data.json’ saved [3676178/3676178]



In [9]:
with open('text_to_sql_data.json') as f:
  base_data = json.load(f)

In [10]:
print(base_data[25]['instruction'])

입력 텍스트: 아프리카에서 활동하는 모든 식량 정의 단체와 그들이 진행한 프로젝트 수를 나열하세요.

DDL statements:
CREATE TABLE food_justice_orgs (org_id INT, org_name TEXT, country TEXT, num_projects INT); INSERT INTO food_justice_orgs (org_id, org_name, country, num_projects) VALUES (1, 'Org A', 'Kenya', 10), (2, 'Org B', 'Nigeria', 7), (3, 'Org C', 'South Africa', 15);

위의 테이블 명세와 사용자의 입력 텍스트를 바탕으로 SQL 쿼리를 작성합니다.


# 함수 모음

In [11]:
def parse_llm_output(llm_output: str, num_pairs: int) -> list[dict]:
    """
    LLM 출력을 파싱해 질문-SQL 딕셔너리를 담은 리스트로 반환

    Args:
        llm_output : LLM 출력 문자열
        num_pairs  : 파싱할 질문-SQL 쌍의 개수

    Returns:
        [{'Question': '...', 'SQL': '...'}, ...]
    """
    questions = re.findall(r'\[질문\]\s*(.*?)\s*(?=\[SQL\])', llm_output, re.DOTALL)
    sqls      = re.findall(r'\[SQL\]\s*(.*?)\s*(?=\[질문\]|$)',  llm_output, re.DOTALL)

    # 백틱 제거
    sqls = [re.sub(r'```sql|```', '', sql).strip() for sql in sqls]
    questions = [q.strip() for q in questions]

    result = []
    for i in range(min(num_pairs, len(questions), len(sqls))):
        result.append({
            'Question': questions[i],
            'SQL':      sqls[i]
        })

    return result

In [12]:
def _convert_to_sqlite(sql: str) -> str:
    """
    DB 종류에 따라 다른 SQL문을 LLM을 사용해 SQLite 문법으로 변환.
    execute_sql_on_db 에서만 사용하는 함수

    Args:
        sql : 변환할 SQL문

    Returns:
        SQLite 문법으로 변환된 SQL문
    """
    response = llm_sql.invoke(
        f"""다음 SQL문을 SQLite 문법으로 변환해줘.
반드시 SQL문만 출력하고 다른 설명은 절대 추가하지 마.
백틱이나 코드블록 없이 순수 SQL문만 출력해.

{sql}"""
    )
    return response.content.strip()


def execute_sql_on_db(parsed_results: list[dict], conn) -> list[dict]:
    """
    질문-SQL 딕셔너리를 담은 List를 받아 DB에 전체 실행해보기.
    """

    results = []

    for item in parsed_results:
        question  = item['Question']
        sql       = item['SQL'].rstrip(';')
        sql_sqlite = None

        # SQL 유형 판별
        sql_type = sql.strip().split()[0].upper()  # SELECT, UPDATE, DELETE, INSERT

        def run_sql(query):
            if sql_type == 'SELECT':
                # SELECT는 pd.read_sql_query() 사용
                return pd.read_sql_query(query, conn), 'success'
            else:
                # INSERT, UPDATE, DELETE는 cursor로 실행 후 롤백
                cursor = conn.cursor()
                cursor.execute(query)
                affected = cursor.rowcount
                conn.rollback()  # 실제 반영 안 되게 롤백
                return pd.DataFrame({'affected_rows': [affected]}), 'success'

        # 1차 시도: 원본 SQL 실행
        try:
            df_result, status = run_sql(sql)
        except Exception as e:
            df_result = None
            status    = f'error: {e}'

            # 2차 시도: LLM으로 SQLite 변환 후 재실행
            print(f"❌ 1차 실행 실패 → LLM으로 SQLite 변환 시도")
            try:
                sql_sqlite        = _convert_to_sqlite(sql).rstrip(';')
                df_result, status = run_sql(sql_sqlite)
            except Exception as e2:
                df_result = None
                status    = f'❌error (변환 후에도 실패): {e2}'

        results.append({
            'Question'  : question,
            'SQL'       : sql,
            'SQL_SQLite': sql_sqlite,
            'Result'    : df_result,
            'Status'    : status
        })

        print(f"1. Question:   {question}")
        print(f"2. SQL 원본:\n{sql}")
        if sql_sqlite:
            print(f"2-1. SQL 변환:\n{sql_sqlite}")
        print(f"3. Status:     {status}")
        print("\n4. 실행 결과:\n")
        print(df_result if df_result is not None else "")
        print("\n")
        print("-" * 50)

    return results

In [13]:
#Prompt 예시 추가용 함수
def get_sample_values(df, n=3) -> str:
  """
  dataframe의 각 칼럼 별 unique한 값 중 랜덤으로 n개를 뽑은 결과 반환. Prompt 추가용.

  인수 :
    dataframe : olist 데이터프레임
    n : 랜덤으로 추출할 값 개수
  return :
    dataframe 각 칼럼에서 unique 한 값 중 랜덤으로 n개를 뽑은 결과 str
  """
  sample_text = ""
  for col in df.columns:
      uniques = df[col].dropna().unique().tolist()
      samples = random.sample(uniques, min(n, len(uniques)))  # 랜덤으로 n개 추출
      sample_text += f"{col} : {samples}\n"
  return sample_text

def get_sample_queries(base_data, n: int):
  """
  json 데이터 중 랜덤으로 n개의 query를 뽑은 결고 반환. Prompt 추가용
  """
  queries = [
    i.get('instruction').split('DDL statements:')[0].split('입력 텍스트:')[1].strip() for i in random.sample(base_data, n)
  ]
  return "\n".join(queries)

In [100]:
def df_to_insert_sql(df: pd.DataFrame, table_name: str) -> str:
    """
    DataFrame을 받아 해당 테이블에 대한 'INSERT INTO ~' SQL 구문만 생성하여 반환한다.

    - 모든 컬럼을 대상으로 VALUES를 작성한다.
    - 삽입되는 행(VALUES) 개수는 1~5개 중 랜덤하게 결정된다.
    - CREATE TABLE 구문은 포함하지 않고 INSERT INTO 부분만 리턴한다.

    Parameters
    ----------
    df : pd.DataFrame - DDL을 만들 대상 데이터
    table_name : str - 테이블 이름

    Returns
    -------
    str
        "INSERT INTO table_name (col1, col2, ...) VALUES (...), (...);" 형태의 문자열
    """
    if df.empty:
        raise ValueError("df가 비어 있어 INSERT 구문을 만들 수 없습니다.")

    columns = list(df.columns)

    # 1~5개 사이에서 랜덤하게 행 개수 결정 (df 행 수보다 클 수 없음)
    num_rows = random.randint(0, min(5, len(df)))
    if num_rows == 0:
      return ''
    sample_df = df.sample(n=num_rows).reset_index(drop=True)

    # 컬럼별로 숫자형인지 여부를 미리 판단 (값 포맷팅에 사용)
    is_numeric_col = {col: pd.api.types.is_numeric_dtype(df[col]) for col in columns}

    def format_value(col, val):
        if pd.isna(val):
            return "NULL"
        if is_numeric_col[col] and not isinstance(val, bool):
            # 정수/실수 그대로 출력
            return str(val)
        if isinstance(val, bool):
            return str(val).upper()  # TRUE / FALSE
        # 문자열, 날짜 등은 작은따옴표로 감싸고 내부 작은따옴표는 이스케이프
        escaped = str(val).replace("'", "''")
        return f"'{escaped}'"

    value_rows = []
    for _, row in sample_df.iterrows():
        values = ", ".join(format_value(col, row[col]) for col in columns)
        value_rows.append(f"({values})")

    columns_str = ", ".join(columns)
    values_str = ", ".join(value_rows)

    return f"INSERT INTO {table_name} ({columns_str}) VALUES {values_str};"



In [14]:
# #최종 저장용 함수
# def convert_to_gretel_format(total_result: list[dict], ddl: str) -> list[dict]:
#     """
#     total_results를 gretelai text-to-sql 학습 데이터 형식으로 변환

#     Args:
#         total_results : [{'Question': ..., 'SQL': ...}, ...] LLM 실행 결과 전체 List
#         ddl           : instruction에 넣을 DDL 문자열

#     Returns:
#         [{'instruction': ..., 'input': '', 'output': ...}, ...]
#     """
#     converted = []
#     for item in total_result:
#         question = item['Question']
#         sql      = item['SQL']

#         instruction = (
#             f"입력 텍스트: {question}\n\n"
#             f"DDL statements:\n{ddl}\n\n"
#             f"위의 테이블 명세와 사용자의 입력 텍스트를 바탕으로 SQL 쿼리를 작성합니다."
#         )

#         converted.append({
#             "instruction": instruction,
#             "input"      : "",
#             "output"     : f"쿼리 작성: {sql}"
#         })

#     return converted

In [63]:
#최종 저장용 함수
def convert_to_gretel_format(total_result: list[dict], ddl: str, df: pd.DataFrame, table_name: str) -> list[dict]:
    """
    total_results를 gretelai text-to-sql 학습 데이터 형식으로 변환

    Args:
        total_results : [{'Question': ..., 'SQL': ...}, ...] LLM 실행 결과 전체 List
        ddl           : instruction에 넣을 DDL 문자열
        df            : INSERT INTO 문을 생성할 DataFrame
        table_name    : INSERT INTO 문에 사용할 테이블 명

    Returns:
        [{'instruction': ..., 'input': '', 'output': ...}, ...]
    """
    converted = []
    for item in total_result:
        question = item['Question']
        sql      = item['SQL']

        insert_sql = df_to_insert_sql(df, table_name)

        instruction = (
            f"입력 텍스트: {question}\n\n"
            f"DDL statements:\n{ddl}\n{insert_sql}\n\n"
            f"위의 테이블 명세와 사용자의 입력 텍스트를 바탕으로 SQL 쿼리를 작성합니다."
        )

        converted.append({
            "instruction": instruction,
            "input"      : "",
            "output"     : f"쿼리 작성: {sql}"
        })

    return converted

In [118]:
import random
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser

ENDING_STYLES = ["~요?", "~까?", "~임?", "~나?", "~습니까?", "~나요?", "~가요?", "~니?", "~냐?"]

REPHRASE_SYSTEM_PROMPT = """당신은 text-to-SQL 데이터셋의 질문(Question) 문장 스타일을 다양화하는 어시스턴트입니다.
주어진 질문을 아래 기준에 맞게 자연스러운 한국어로 다시 작성하세요.

[기준 1] 컬럼명 직접/간접 언급
질문에 영문 컬럼명이나 변수명이 그대로 노출되어 있다면, 의미가 통하는 자연어 표현으로 바꾸세요.
예) "각 country_of_origin별 모든 satellites의 최대 거리는 얼마인가요?"
    -> "각 국가별로 지구 표면으로부터 모든 위성의 최대 거리는 얼마인가요?"
예) "country가 Africa인 모든 org_name 값과 그들이 진행한 num_projects 수를 나열하세요"
    -> "아프리카에서 활동하는 모든 식량 정의 단체와 그들이 진행한 프로젝트 수를 나열하세요."

[기준 2] 명사구 형태 변형 (지시가 있을 때만)
완전한 문장(서술어로 끝남) 대신, 명사구로 끝나는 형태로 바꾸세요.
예) "마을 변호사는 몇 명이었는가?" -> "마을변호사 인원 수"
예) "각 고객별 첫 구매 일시를 알고 싶습니다. 고객 ID와 첫 구매 타임스탬프를 반환해 주세요."
    -> "각 고객별 첫 구매 일시에 대한 고객 ID와 첫 구매 타임스탬프."
예) "2018년 2분기(Q2)에 구매된 주문들의 구매 시각부터 배송사 인계까지 평균 며칠이 걸렸는지 알려주세요"
    -> "2018년 2분기 구매 시각부터 배송사 인계까지 평균일."

[기준 3] 문장 종결 어미 변경 (문장형일 때만)
지정된 종결 어미를 사용해 문장을 끝맺으세요. (~요?, ~까?, ~임?, ~나? 등)

[중요]
- 질문의 의미(조건, 대상 컬럼, 집계/필터 방식 등)는 절대 바꾸지 마세요. SQL과 매칭이 깨지면 안 됩니다.
- 결과는 변형된 질문 문장 하나만 출력하세요. 따옴표, 설명, 번호, "변형된 질문:" 같은 접두사를 붙이지 마세요."""

USER_PROMPT = """원본 질문: {question}

다음 지시를 반영해 질문을 다시 작성하세요.
- 명사구 형태로 변형: {use_noun_phrase}
- 문장으로 끝낼 경우 종결 어미: {ending_style}"""


def rephrase_questions(total_result: list[dict], llm, noun_phrase_ratio: float = 0.4) -> list[dict]:
    """
    total_result의 Question만 LLM을 통해 다양한 말투/표현으로 바꿔서 반환한다. (SQL은 그대로 유지)

    Args:
        total_result      : [{'Question': ..., 'SQL': ...}, ...]
        llm                : LangChain ChatModel (예: llm_small)
        noun_phrase_ratio  : 명사구 형태로 변형할 확률 (기본 40%)

    Returns:
        [{'Question': 변형된 질문, 'SQL': 원본 SQL}, ...]
    """
    prompt = ChatPromptTemplate.from_messages([
        ("system", REPHRASE_SYSTEM_PROMPT),
        ("human", USER_PROMPT),
    ])
    chain = prompt | llm | StrOutputParser()

    # row마다 명사구 여부 / 종결어미를 랜덤하게 지정해서 batch 입력 구성
    batch_inputs = []
    for item in total_result:
        use_noun_phrase = random.random() < noun_phrase_ratio
        ending_style = random.choice(ENDING_STYLES)
        batch_inputs.append({
            "question": item["Question"],
            "use_noun_phrase": "예 (명사구로 끝내기)" if use_noun_phrase else "아니오 (완전한 문장 유지)",
            "ending_style": "해당 없음 (명사구이므로 종결어미 사용 안 함)" if use_noun_phrase else ending_style,
        })

    # LangChain batch 호출로 한 번에 처리 (내부적으로 병렬 실행)
    try:
        rephrased_questions = chain.batch(batch_inputs, config={"max_concurrency": 5})
    except Exception as e:
        print(f"⚠️ batch 호출 중 오류 발생, 개별 호출로 재시도: {e}")
        rephrased_questions = []
        for inp in batch_inputs:
            try:
                rephrased_questions.append(chain.invoke(inp))
            except Exception:
                rephrased_questions.append(None)

    converted = []
    for item, new_question in zip(total_result, rephrased_questions):
        # 실패한 경우 원본 질문 유지
        question = new_question.strip() if new_question else item["Question"]
        converted.append({
            "Question": question,
            "SQL": item["SQL"],
        })
    return converted

# LLM 관련 설정

In [16]:
SYSTEM_PROMPT = """
#역할
당신은 Text-to-SQL을 수행해야합니다.
DDL 선언문, 칼럼 설명, 칼럼 값 예시, 질문 예시를 참고해 사용자가 할 법한 질문-SQL 쌍을 작성해주세요.
실제 사용자가 Text-to-SQL LLM에게 자연스럽게 물어볼 법한 질문과 그에 정확히 대응하는 SQL을 작성하세요.
질문-SQL쌍은 10개 생성하세요.

#최우선 중요 원칙
사람이 실제로 어떻게 질문할지 생각하세요. 그리고 그 질문에 정확히 대응하는 SQL 문을 작성하세요.
질문 예시를 적극적으로 참고하세요.

#규칙
1. 반드시 코드 블록 없이 순수 SQL만 출력하세요.
2. history를 참고해 중복이 없게끔 하세요.
3. 다음 리스트 같은 질문은 비현실적인 질문입니다
- '2017-03-14 12:58:42'에 구매된 주문의 배송 예정일을 '2018-11-01'로 업데이트하고 싶습니다. : 사람은 날짜 단위를 시분초까지 쪼개서 요청하지 않습니다.
4. SQL 작성시 주의
- BETWEEN : 기간을 조회할 때 BETWEEN으로 끝 날짜를 지정하면 마지막 날이 누락됩니다. 기간 조회는 ">= 시작일 AND < 다음 기간 시작일" 패턴을, 하루 조회는 DATE() 함수를 사용하세요.
   - 나쁨: WHERE col BETWEEN '2018-06-01' AND '2018-06-30'  (6월 30일 누락)
   - 좋음: WHERE col >= '2018-06-01' AND col < '2018-07-01'
   - 좋음: WHERE DATE(col) = '2018-06-04'
- 현재 날짜/시간 : 현재 시각이나 오늘 날짜에 의존하는 질문·SQL을 만들지 마세요.
"지금", "오늘", "최근 1년", "이번 달" 처럼 실행 시점에 따라 답이 달라지는 표현을 쓰지 마세요.
NOW(), CURRENT_DATE, CURRENT_TIMESTAMP, DATE_ADD/SUB(NOW()...) 같은 함수도 사용 금지입니다.
날짜 조건은 '2018-05-01' 처럼 고정된 날짜 리터럴로만 작성하세요.

#출력 형식
반드시 다음 형식을 지켜 출력해주세요.
[질문]
자연어 질문
[SQL]
SQL문
"""

In [17]:
SYSTEM_PROMPT_MULTI = """
#역할
당신은 text-to-SQL을 수행해야 합니다.
아래에 제공되는 2개에 DDL 선언문, 칼럼 설명, 칼럼별 값 예시를 참고해 사용자의 입장에서 할 수 있는 자연어 질문과 그에 대응하는 SQL문을 작성해주세요.
질문-SQL 쌍을 3개 생성하세요.

#규칙
1. 반드시 코드 블록 없이 순수 SQL만 출력하세요.
2. history를 참고해 중복이 없게끔 하세요.
3. 모든 질문-SQL 쌍은 반드시 두 테이블을 모두 사용해야 합니다.
   따라서 SQL은 SELECT 문으로 작성하며, 아래 패턴을 고르게 사용하세요.
   - 크로스 테이블 조인: 두 테이블을 JOIN (INNER, LEFT, RIGHT, FULL OUTER)
   - 집합 연산: 두 테이블의 결과를 UNION / UNION ALL / INTERSECT / EXCEPT로 결합
   - 서브쿼리: 한 테이블의 결과를 다른 테이블 조건으로 사용 (IN, EXISTS 등)
4. 칼럼별 값 예시를 활용해 구체적인 값을 포함한 질문을 생성하세요.

#주의
1. 다음 리스트 같은 질문은 비현실적인 질문입니다
- '2017-03-14 12:58:42'에 구매된 주문의 배송 예정일을 '2018-11-01'로 업데이트하고 싶습니다. : 사람은 날짜 단위를 시분초까지 쪼개서 요청하지 않습니다.

2. SQL 작성 시 주의
   기간을 조회할 때 BETWEEN으로 끝 날짜를 지정하면 마지막 날이 누락됩니다.
   기간 조회는 ">= 시작일 AND < 다음 기간 시작일" 패턴을, 하루 조회는 DATE() 함수를 사용하세요.
   - 나쁨: WHERE col BETWEEN '2018-06-01' AND '2018-06-30'  (6월 30일 누락)
   - 좋음: WHERE col >= '2018-06-01' AND col < '2018-07-01'
   - 좋음: WHERE DATE(col) = '2018-06-04'

3. 현재 날짜/시간
현재 시각이나 오늘 날짜에 의존하는 질문·SQL을 만들지 마세요.
"지금", "오늘", "최근 1년", "이번 달" 처럼 실행 시점에 따라 답이 달라지는 표현을 쓰지 마세요.
NOW(), CURRENT_DATE, CURRENT_TIMESTAMP, DATE_ADD/SUB(NOW()...) 같은 함수도 사용 금지입니다.
날짜 조건은 '2018-05-01' 처럼 고정된 날짜 리터럴로만 작성하세요.

#출력 형식
반드시 다음 형식을 지켜 출력해주세요.
[질문]
자연어 질문
[SQL]
SQL문
"""

In [18]:
HUMAN_PROMPT = """
# DDL 선언문
{DDL_Statement}

# 칼럼 설명
{column_descriptions}

# 칼럼 값 예시
{column_values}

#질문 예시
{question_examples}

# history
{history}
"""

In [19]:
def generate_sql(DDL_Statement: str, column_descriptions: str, column_values: str, question_examples: str,
                 history : list[dict], is_multi=False) -> str:
    """
    DDL문과 컬럼 설명을 입력받아 LLM으로 질문-SQL 쌍을 생성

    Args:
        DDL_Statement       : 테이블 DDL 문
        column_descriptions : 컬럼 설명
        column_values       : 컬럼 값 예시 (함수 사용해서 들어감)
        question_examples   : 질문 예시들

    Returns:
        LLM이 생성한 질문-SQL 쌍 문자열
    """

    system_prompt = SYSTEM_PROMPT_MULTI if is_multi else SYSTEM_PROMPT

    prompt = ChatPromptTemplate.from_messages([
        SystemMessagePromptTemplate.from_template(system_prompt),
        HumanMessagePromptTemplate.from_template(HUMAN_PROMPT),
    ])

    chain = prompt | llm

    # history 중 질문만 str로 취합
    history_str = ""
    for res in total_result:
      history_str += res.get('Question') + "\n"

    response = chain.invoke({
        "DDL_Statement"      : DDL_Statement,
        "column_descriptions": column_descriptions,
        "column_values" : column_values,
        "question_examples" : question_examples,
        "history" : history_str
    })

    print(response.usage_metadata) # 캐싱 작동 확인용
    return response.content

#단일 SQL 문 생성


각 DDL문은 다음 프롬프트를 사용해 얻기.
(Human Prompt에 DF으로부터 값 예시 랜덤으로 넣는 로직 삭제하고 DDL 문의 INSERT INTO ~ VALUES로 대체하기)

```
DB에 대한 DDL문을 다음 예시처럼 한 줄로 작성해줘.
DDL문을 제외한 그 어떤 출력도 하지마.
DDL문만 출력해줘.

#테이블명
orders

#DB
order_id	customer_id	order_status	order_purchase_timestamp	order_approved_at	order_delivered_carrier_date	order_delivered_customer_date	order_estimated_delivery_date
e481f51cbdc54678b7cc49136f2d6af7	9ef432eb6251297304e76186b10a928d	delivered	2017-10-02 10:56:33	2017-10-02 11:07:15	2017-10-04 19:55:00	2017-10-10 21:25:13	2017-10-18 00:00:00
53cdb2fc8bc7dce0b6741e2150273451	b0830fb4747a6c6d20dea0b8c802d7ef	delivered	2018-07-24 20:41:37	2018-07-26 03:24:27	2018-07-26 14:31:00	2018-08-07 15:27:45	2018-08-13 00:00:00
47770eb9100c2d0c44946d9cf07ec65d	41ce2a54c0b03bf3443c3d931a367089	delivered	2018-08-08 08:38:49	2018-08-08 08:55:23	2018-08-08 13:50:00	2018-08-17 18:06:29	2018-09-04 00:00:00

#예시
CREATE TABLE salesperson (salesperson_id INT, name TEXT, region TEXT); INSERT INTO salesperson (salesperson_id, name, region) VALUES (1, 'John Doe', 'North'), (2, 'Jane Smith', 'South'); CREATE TABLE timber_sales (sales_id INT, salesperson_id INT, volume REAL, sale_date DATE); INSERT INTO timber_sales (sales_id, salesperson_id, volume, sale_date) VALUES (1, 1, 120, '2021-01-01'), (2, 1, 150, '2021-02-01'), (3, 2, 180, '2021-01-01');
CREATE TABLE farmers_india (id INT, name VARCHAR(255), district_id INT, age INT, income INT); INSERT INTO farmers_india (id, name, district_id, age, income) VALUES (1, 'Farmer A', 1, 45, 50000); CREATE TABLE districts_india (id INT, name VARCHAR(255), state VARCHAR(255)); INSERT INTO districts_india (id, name, state) VALUES (1, 'District A', 'Maharashtra');
CREATE TABLE Armed_Forces (base_id INT, base_name VARCHAR(50), base_location VARCHAR(50), base_type VARCHAR(50)); INSERT INTO Armed_Forces (base_id, base_name, base_location, base_type) VALUES (1, 'Fort Bragg', 'North Carolina', 'Army'); INSERT INTO Armed_Forces (base_id, base_name, base_location, base_type) VALUES (2, 'Camp Pendleton', 'California', 'Marines');
```

## 1.olist_orders_dataset

In [138]:
# 변수 설정 (df마다 1,2,3,4 수정해야함.)
DF_NAME = df_orders
TABLE_NAME = "orders"

  #1. DDL 선언문 : INSERT INTO VALUES 삭제
DDL_Statement = """
CREATE TABLE orders (order_id VARCHAR(32) NOT NULL, customer_id VARCHAR(32) NOT NULL, order_status VARCHAR(20) NOT NULL, order_purchase_timestamp DATETIME NOT NULL, order_approved_at DATETIME NULL, order_delivered_carrier_date DATETIME NULL, order_delivered_customer_date DATETIME NULL, order_estimated_delivery_date DATETIME NOT NULL, PRIMARY KEY (order_id));
"""

  #2. 칼럼 설명 (README 참고)
column_descriptions = """
order_id : unique identifier of the order.
customer_id : key to the customer dataset. Each order has a unique customer_id.
order_status : Reference to the order status (delivered, shipped, etc)
order_purchase_timestamp : Shows the purchase timestamp.
order_approved_at : Shows the payment approval timestamp.
order_delivered_carrier_date : Shows the order posting timestamp. When it was handled to the logistic partner.
order_delivered_customer_date : Shows the actual order delivery date to the customer.
order_estimated_delivery_date : Shows the estimated delivery date that was informed to customer at the purchase moment.
"""

#LLM 실행
total_result = []
for i in range(10):
  print(f'진행중 : {i+1}')

  #3. 칼럼 별 unique한 값 예시
  column_values = get_sample_values(DF_NAME, 3)

  #4. base dataset으로부터 질문 예시 n개 추출
  question_examples = get_sample_queries(base_data, 10)

  result = generate_sql(DDL_Statement = DDL_Statement,
                        column_descriptions = column_descriptions,
                        column_values = column_values,
                        question_examples = question_examples,
                        history = total_result)

  #LLM 응답 파싱 후 저장. (n = LLM이 한 번에 생성하는 질문-SQL쌍 개수 (Prompt에 적혀있는데 따로 수정할 필요는 없는거같아서 냅둠.))
  parsed = parse_llm_output(result, 10)
  total_result.extend(parsed)


#후처리
#질문 말투 다양화
total_result = rephrase_questions(total_result, llm_small, 0.4)

#결과 저장하기.
converted_data = convert_to_gretel_format(total_result, DDL_Statement, DF_NAME, TABLE_NAME)

os.makedirs('data', exist_ok=True)
with open(f'data/Olist_{TABLE_NAME}_text_to_sql_data.json', 'w', encoding='utf-8') as f:
    json.dump(converted_data, f, ensure_ascii=False, indent=4)

print(f"✅ {len(converted_data)}개 변환 완료!")

진행중 : 1
{'input_tokens': 1367, 'output_tokens': 3787, 'total_tokens': 5154, 'input_token_details': {'audio': 0, 'cache_read': 0}, 'output_token_details': {'audio': 0, 'reasoning': 3008}}
진행중 : 2
{'input_tokens': 1603, 'output_tokens': 4413, 'total_tokens': 6016, 'input_token_details': {'audio': 0, 'cache_read': 0}, 'output_token_details': {'audio': 0, 'reasoning': 3584}}
진행중 : 3
{'input_tokens': 1876, 'output_tokens': 4611, 'total_tokens': 6487, 'input_token_details': {'audio': 0, 'cache_read': 0}, 'output_token_details': {'audio': 0, 'reasoning': 3840}}
진행중 : 4
{'input_tokens': 2050, 'output_tokens': 4590, 'total_tokens': 6640, 'input_token_details': {'audio': 0, 'cache_read': 0}, 'output_token_details': {'audio': 0, 'reasoning': 3712}}
진행중 : 5
{'input_tokens': 2394, 'output_tokens': 4439, 'total_tokens': 6833, 'input_token_details': {'audio': 0, 'cache_read': 0}, 'output_token_details': {'audio': 0, 'reasoning': 3456}}
진행중 : 6
{'input_tokens': 2624, 'output_tokens': 4127, 'total_toke

In [139]:
# 파싱된 SQL 전체 DB에 실행해보기.
conn = sqlite3.connect(':memory:') #원래는 sqlite3.connect('mydb.db')로 Disk에 저장. 지금은 :memory:로 RAM에 저장
DF_NAME.to_sql(TABLE_NAME, conn, if_exists='replace', index=False)
execute_res = execute_sql_on_db(total_result, conn)

1. Question:   2018년 6월에 구매된 주문 건수
2. SQL 원본:
SELECT COUNT(*) AS order_count
FROM orders
WHERE order_purchase_timestamp >= '2018-06-01' AND order_purchase_timestamp < '2018-07-01'
3. Status:     success

4. 실행 결과:

   order_count
0         6167


--------------------------------------------------
1. Question:   주문 상태별 주문 건수를 알려주나?
2. SQL 원본:
SELECT order_status, COUNT(*) AS cnt
FROM orders
GROUP BY order_status
ORDER BY cnt DESC
3. Status:     success

4. 실행 결과:

  order_status    cnt
0    delivered  96478
1      shipped   1107
2     canceled    625
3  unavailable    609
4     invoiced    314
5   processing    301
6      created      5
7     approved      2


--------------------------------------------------
1. Question:   2017년 12월 25일에 고객에게 배송 완료된 주문의 주문 ID와 고객 ID를 보여주는 것임?
2. SQL 원본:
SELECT order_id, customer_id
FROM orders
WHERE DATE(order_delivered_customer_date) = '2017-12-25'
3. Status:     success

4. 실행 결과:

Empty DataFrame
Columns: [order_id, customer_id]
Index: []


-------

KeyboardInterrupt: 

In [140]:
#질문 출력해보기
for result in total_result:
  print(result.get('Question'))

2018년 6월에 구매된 주문 건수
주문 상태별 주문 건수를 알려주나?
2017년 12월 25일에 고객에게 배송 완료된 주문의 주문 ID와 고객 ID를 보여주는 것임?
예상 배송일을 넘겨서 도착한 주문 건수
2017년 5월에 구매된 주문들의 결제 승인까지 평균 소요 시간(시간 단위)은 얼마인가요?
2018년 1분기에 구매된 주문 중 배송사에 인계된 주문의 비율은 얼마인가요?
각 고객별 첫 구매 일시는 언제인가요?
2017년에 고객에게 배송 완료된 주문들의 평균 배송 소요일수.
2018년 2월에 고객에게 배송 완료된 주문 중 예상 배송일보다 빨리 도착한 주문의 비율은 얼마인가요?
2017년 9월 10일에 배송사로 인계된 주문의 주문 ID와 인계 시각을 보여주냐?
2017년 상반기에 가장 많이 주문한 상위 5명의 고객 ID와 주문 건수를 알려주실 수 있나요?
2018년 3월에 구매된 주문들의 결제 승인부터 배송사 인계까지 평균 소요 시간(시간 단위)은 얼마입니까?
2018-07-15에 결제 승인됐지만 고객에게 아직 배송 완료되지 않은 주문의 주문 ID를 보여주겠니?
2018년 2분기(4~6월)에 구매된 주문의 상태별 건수를 알려주니?
2017년에 고객에게 배송 완료된 주문 중 예상 배송일을 초과한 주문의 평균 지연 일수
결제 승인 시각보다 먼저 배송사에 인계된 이상 주문은 몇 건인가요?
2017년 월별 구매 건수
고객 수령까지 가장 오래 걸린 주문 상위 10건의 주문 ID와 소요일수(일)을 알려주시나요?
주문 상태가 'unavailable'인 주문들의 주문 ID와 구매 일시를 최신순으로 보여주나요?
2017년 11월 11일에 구매된 주문은 총 몇 건인가요?
2018년 7월에 결제 승인이 난 주문 건수
배송사에 인계됐지만 아직 고객에게 배송 완료되지 않은 주문 건수는 얼마인가요?
2017년 8월 1일부터 10일까지 구매된 주문 중 고객에게 배송 완료된 주문의 비율
예상 배송일보다 빨리 도착한 주문들의 평균 조기 도착 일수는 얼마인가?
2017년 4분기(10~12월)

##2.olist_order_items_dataset

In [144]:
# 변수 설정 (df마다 1,2,3,4 수정해야함.)
DF_NAME = df_order_items
TABLE_NAME = "order_items"

  #1. DDL 선언문 : INSERT INTO VALUES 삭제
DDL_Statement = """
CREATE TABLE order_items (order_id VARCHAR(32) NOT NULL, order_item_id INT NOT NULL, product_id VARCHAR(32) NOT NULL, seller_id VARCHAR(32) NOT NULL, shipping_limit_date DATETIME NOT NULL, price DECIMAL(10,2) NOT NULL, freight_value DECIMAL(10,2) NOT NULL, PRIMARY KEY (order_id, order_item_id));
"""

  #2. 칼럼 설명 (README 참고)
column_descriptions = """
order_id : order unique identifier
order_item_id : sequential number identifying number of items included in the same order.
product_id : product unique identifier
seller_id : seller unique identifier
shipping_limit_date : Shows the seller shipping limit date for handling the order over to the logistic partner.
price : item price
freight_value ; item freight value item (if an order has more than one item the freight value is splitted between items)
"""

#LLM 실행
total_result = []
for i in range(10):
  print(f'진행중 : {i+1}')

  #3. 칼럼 별 unique한 값 예시
  column_values = get_sample_values(DF_NAME, 3)

  #4. base dataset으로부터 질문 예시 n개 추출
  question_examples = get_sample_queries(base_data, 10)

  result = generate_sql(DDL_Statement = DDL_Statement,
                        column_descriptions = column_descriptions,
                        column_values = column_values,
                        question_examples = question_examples,
                        history = total_result)

  #LLM 응답 파싱 후 저장. (n = LLM이 한 번에 생성하는 질문-SQL쌍 개수 (Prompt에 적혀있는데 따로 수정할 필요는 없는거같아서 냅둠.))
  parsed = parse_llm_output(result, 10)
  total_result.extend(parsed)


#후처리
#질문 말투 다양화
total_result = rephrase_questions(total_result, llm_small, 0.4)

#결과 저장하기.
converted_data = convert_to_gretel_format(total_result, DDL_Statement, DF_NAME, TABLE_NAME)

os.makedirs('data', exist_ok=True)
with open(f'data/Olist_{TABLE_NAME}_text_to_sql_data.json', 'w', encoding='utf-8') as f:
    json.dump(converted_data, f, ensure_ascii=False, indent=4)

print(f"✅ {len(converted_data)}개 변환 완료!")

진행중 : 1
{'input_tokens': 1261, 'output_tokens': 4072, 'total_tokens': 5333, 'input_token_details': {'audio': 0, 'cache_read': 0}, 'output_token_details': {'audio': 0, 'reasoning': 3136}}
진행중 : 2
{'input_tokens': 1565, 'output_tokens': 3564, 'total_tokens': 5129, 'input_token_details': {'audio': 0, 'cache_read': 0}, 'output_token_details': {'audio': 0, 'reasoning': 2560}}
진행중 : 3
{'input_tokens': 1854, 'output_tokens': 4668, 'total_tokens': 6522, 'input_token_details': {'audio': 0, 'cache_read': 0}, 'output_token_details': {'audio': 0, 'reasoning': 3776}}
진행중 : 4
{'input_tokens': 2156, 'output_tokens': 4731, 'total_tokens': 6887, 'input_token_details': {'audio': 0, 'cache_read': 0}, 'output_token_details': {'audio': 0, 'reasoning': 3904}}
진행중 : 5
{'input_tokens': 2419, 'output_tokens': 4967, 'total_tokens': 7386, 'input_token_details': {'audio': 0, 'cache_read': 0}, 'output_token_details': {'audio': 0, 'reasoning': 3968}}
진행중 : 6
{'input_tokens': 2755, 'output_tokens': 5212, 'total_toke

##3.olist_order_reviews_dataset

In [152]:
# 변수 설정 (df마다 1,2,3,4 수정해야함.)
DF_NAME = df_order_reviews
TABLE_NAME = "order_reviews"

  #1. DDL 선언문 : INSERT INTO VALUES 삭제
DDL_Statement = """
CREATE TABLE order_reviews (review_id VARCHAR(32) NOT NULL, order_id VARCHAR(32) NOT NULL, review_score INT NOT NULL, review_comment_title VARCHAR(100) NULL, review_comment_message TEXT NULL, review_creation_date DATETIME NOT NULL, review_answer_timestamp DATETIME NOT NULL, PRIMARY KEY (review_id));
"""

  #2. 칼럼 설명 (README 참고)
column_descriptions = """
review_id : unique review identifier
order_id : unique order identifier
review_score : Note ranging from 1 to 5 given by the customer on a satisfaction survey.
review_comment_title : Comment title from the review left by the customer, in Portuguese.
review_comment_message : Comment message from the review left by the customer, in Portuguese.
review_creation_date : Shows the date in which the satisfaction survey was sent to the customer.
review_answer_timestamp : Shows satisfaction survey answer timestamp.
"""

#LLM 실행
total_result = []
for i in range(10):
  print(f'진행중 : {i+1}')

  #3. 칼럼 별 unique한 값 예시
  column_values = get_sample_values(DF_NAME, 3)

  #4. base dataset으로부터 질문 예시 n개 추출
  question_examples = get_sample_queries(base_data, 10)

  result = generate_sql(DDL_Statement = DDL_Statement,
                        column_descriptions = column_descriptions,
                        column_values = column_values,
                        question_examples = question_examples,
                        history = total_result)

  #LLM 응답 파싱 후 저장. (n = LLM이 한 번에 생성하는 질문-SQL쌍 개수 (Prompt에 적혀있는데 따로 수정할 필요는 없는거같아서 냅둠.))
  parsed = parse_llm_output(result, 10)
  total_result.extend(parsed)


#후처리
#질문 말투 다양화
total_result = rephrase_questions(total_result, llm_small, 0.4)

#결과 저장하기.
converted_data = convert_to_gretel_format(total_result, DDL_Statement, DF_NAME, TABLE_NAME)

os.makedirs('data', exist_ok=True)
with open(f'data/Olist_{TABLE_NAME}_text_to_sql_data.json', 'w', encoding='utf-8') as f:
    json.dump(converted_data, f, ensure_ascii=False, indent=4)

print(f"✅ {len(converted_data)}개 변환 완료!")

진행중 : 1
{'input_tokens': 1303, 'output_tokens': 4510, 'total_tokens': 5813, 'input_token_details': {'audio': 0, 'cache_read': 0}, 'output_token_details': {'audio': 0, 'reasoning': 3648}}
진행중 : 2
{'input_tokens': 1551, 'output_tokens': 6032, 'total_tokens': 7583, 'input_token_details': {'audio': 0, 'cache_read': 0}, 'output_token_details': {'audio': 0, 'reasoning': 4992}}
진행중 : 3
{'input_tokens': 1958, 'output_tokens': 6378, 'total_tokens': 8336, 'input_token_details': {'audio': 0, 'cache_read': 0}, 'output_token_details': {'audio': 0, 'reasoning': 5312}}
진행중 : 4
{'input_tokens': 2146, 'output_tokens': 5902, 'total_tokens': 8048, 'input_token_details': {'audio': 0, 'cache_read': 0}, 'output_token_details': {'audio': 0, 'reasoning': 4800}}
진행중 : 5
{'input_tokens': 2447, 'output_tokens': 4824, 'total_tokens': 7271, 'input_token_details': {'audio': 0, 'cache_read': 0}, 'output_token_details': {'audio': 0, 'reasoning': 4032}}
진행중 : 6
{'input_tokens': 2656, 'output_tokens': 6432, 'total_toke

##4. olist_order_payments_dataset

In [160]:
# 변수 설정 (df마다 1,2,3,4 수정해야함.)
DF_NAME = df_order_payments
TABLE_NAME = "order_payments"

  #1. DDL 선언문 : INSERT INTO VALUES 삭제
DDL_Statement = """
CREATE TABLE order_payments (order_id VARCHAR(32) NOT NULL, payment_sequential INT NOT NULL, payment_type VARCHAR(20) NOT NULL, payment_installments INT NOT NULL, payment_value DECIMAL(10,2) NOT NULL, PRIMARY KEY (order_id, payment_sequential));
"""

  #2. 칼럼 설명 (README 참고)
column_descriptions = """
order_id : unique identifier of an order.
payment_sequential : a customer may pay an order with more than one payment method. If he does so, a sequence will be created to accommodate all payments.
payment_type : method of payment chosen by the customer.
payment_installments : number of installments chosen by the customer.
payment_value : transaction value.
"""

#LLM 실행
total_result = []
for i in range(10):
  print(f'진행중 : {i+1}')

  #3. 칼럼 별 unique한 값 예시
  column_values = get_sample_values(DF_NAME, 3)

  #4. base dataset으로부터 질문 예시 n개 추출
  question_examples = get_sample_queries(base_data, 10)

  result = generate_sql(DDL_Statement = DDL_Statement,
                        column_descriptions = column_descriptions,
                        column_values = column_values,
                        question_examples = question_examples,
                        history = total_result)

  #LLM 응답 파싱 후 저장. (n = LLM이 한 번에 생성하는 질문-SQL쌍 개수 (Prompt에 적혀있는데 따로 수정할 필요는 없는거같아서 냅둠.))
  parsed = parse_llm_output(result, 10)
  total_result.extend(parsed)


#후처리
#질문 말투 다양화
total_result = rephrase_questions(total_result, llm_small, 0.4)

#결과 저장하기.
converted_data = convert_to_gretel_format(total_result, DDL_Statement, DF_NAME, TABLE_NAME)

os.makedirs('data', exist_ok=True)
with open(f'data/Olist_{TABLE_NAME}_text_to_sql_data.json', 'w', encoding='utf-8') as f:
    json.dump(converted_data, f, ensure_ascii=False, indent=4)

print(f"✅ {len(converted_data)}개 변환 완료!")

진행중 : 1
{'input_tokens': 1099, 'output_tokens': 3381, 'total_tokens': 4480, 'input_token_details': {'audio': 0, 'cache_read': 0}, 'output_token_details': {'audio': 0, 'reasoning': 2560}}
진행중 : 2
{'input_tokens': 1383, 'output_tokens': 5347, 'total_tokens': 6730, 'input_token_details': {'audio': 0, 'cache_read': 0}, 'output_token_details': {'audio': 0, 'reasoning': 4352}}
진행중 : 3
{'input_tokens': 1652, 'output_tokens': 5841, 'total_tokens': 7493, 'input_token_details': {'audio': 0, 'cache_read': 0}, 'output_token_details': {'audio': 0, 'reasoning': 4800}}
진행중 : 4
{'input_tokens': 1960, 'output_tokens': 5771, 'total_tokens': 7731, 'input_token_details': {'audio': 0, 'cache_read': 0}, 'output_token_details': {'audio': 0, 'reasoning': 4864}}
진행중 : 5
{'input_tokens': 2290, 'output_tokens': 5748, 'total_tokens': 8038, 'input_token_details': {'audio': 0, 'cache_read': 0}, 'output_token_details': {'audio': 0, 'reasoning': 4672}}
진행중 : 6
{'input_tokens': 2609, 'output_tokens': 5732, 'total_toke

##5.olist_products_dataset

In [169]:
# 변수 설정 (df마다 1,2,3,4 수정해야함.)
DF_NAME = df_products
TABLE_NAME = "products"

  #1. DDL 선언문 : INSERT INTO VALUES 삭제
DDL_Statement = """
CREATE TABLE products (product_id VARCHAR(32) NOT NULL, product_category_name VARCHAR(50) NULL, product_name_lenght INT NULL, product_description_lenght INT NULL, product_photos_qty INT NULL, product_weight_g INT NULL, product_length_cm INT NULL, product_height_cm INT NULL, product_width_cm INT NULL, PRIMARY KEY (product_id));
"""

  #2. 칼럼 설명 (README 참고)
column_descriptions = """
product_id : unique product identifier
product_category_name : root category of product, in Portuguese.
product_name_lenght : number of characters extracted from the product name.
product_description_lenght : number of characters extracted from the product description.
product_photos_qty : number of product published photos
product_weight_g : product weight measured in grams.
product_length_cm : product length measured in centimeters.
product_height_cm : product height measured in centimeters.
product_width_cm : product width measured in centimeters.
"""

#LLM 실행
total_result = []
for i in range(10):
  print(f'진행중 : {i+1}')

  #3. 칼럼 별 unique한 값 예시
  column_values = get_sample_values(DF_NAME, 3)

  #4. base dataset으로부터 질문 예시 n개 추출
  question_examples = get_sample_queries(base_data, 10)

  result = generate_sql(DDL_Statement = DDL_Statement,
                        column_descriptions = column_descriptions,
                        column_values = column_values,
                        question_examples = question_examples,
                        history = total_result)

  #LLM 응답 파싱 후 저장. (n = LLM이 한 번에 생성하는 질문-SQL쌍 개수 (Prompt에 적혀있는데 따로 수정할 필요는 없는거같아서 냅둠.))
  parsed = parse_llm_output(result, 10)
  total_result.extend(parsed)


#후처리
#질문 말투 다양화
total_result = rephrase_questions(total_result, llm_small, 0.4)

#결과 저장하기.
converted_data = convert_to_gretel_format(total_result, DDL_Statement, DF_NAME, TABLE_NAME)

os.makedirs('data', exist_ok=True)
with open(f'data/Olist_{TABLE_NAME}_text_to_sql_data.json', 'w', encoding='utf-8') as f:
    json.dump(converted_data, f, ensure_ascii=False, indent=4)

print(f"✅ {len(converted_data)}개 변환 완료!")

진행중 : 1
{'input_tokens': 1193, 'output_tokens': 3619, 'total_tokens': 4812, 'input_token_details': {'audio': 0, 'cache_read': 0}, 'output_token_details': {'audio': 0, 'reasoning': 2816}}
진행중 : 2
{'input_tokens': 1444, 'output_tokens': 4119, 'total_tokens': 5563, 'input_token_details': {'audio': 0, 'cache_read': 0}, 'output_token_details': {'audio': 0, 'reasoning': 3072}}
진행중 : 3
{'input_tokens': 1724, 'output_tokens': 5750, 'total_tokens': 7474, 'input_token_details': {'audio': 0, 'cache_read': 0}, 'output_token_details': {'audio': 0, 'reasoning': 4736}}
진행중 : 4
{'input_tokens': 1936, 'output_tokens': 5082, 'total_tokens': 7018, 'input_token_details': {'audio': 0, 'cache_read': 0}, 'output_token_details': {'audio': 0, 'reasoning': 4032}}
진행중 : 5
{'input_tokens': 2328, 'output_tokens': 5655, 'total_tokens': 7983, 'input_token_details': {'audio': 0, 'cache_read': 0}, 'output_token_details': {'audio': 0, 'reasoning': 4800}}
진행중 : 6
{'input_tokens': 2623, 'output_tokens': 4347, 'total_toke

##6.olist_sellers_dataset

In [171]:
# 변수 설정 (df마다 1,2,3,4 수정해야함.)
DF_NAME = df_sellers
TABLE_NAME = "sellers"

  #1. DDL 선언문 : INSERT INTO VALUES 삭제
DDL_Statement = """
CREATE TABLE sellers (seller_id VARCHAR(32) NOT NULL, seller_zip_code_prefix VARCHAR(5) NOT NULL, seller_city VARCHAR(40) NOT NULL, seller_state VARCHAR(2) NOT NULL, PRIMARY KEY (seller_id));
"""

  #2. 칼럼 설명 (README 참고)
column_descriptions = """
seller_id : seller unique identifier
seller_zip_code_prefix : first 5 digits of seller zip code
seller_city : seller city name
seller_state : seller state
"""

#LLM 실행
total_result = []
for i in range(10):
  print(f'진행중 : {i+1}')

  #3. 칼럼 별 unique한 값 예시
  column_values = get_sample_values(DF_NAME, 3)

  #4. base dataset으로부터 질문 예시 n개 추출
  question_examples = get_sample_queries(base_data, 10)

  result = generate_sql(DDL_Statement = DDL_Statement,
                        column_descriptions = column_descriptions,
                        column_values = column_values,
                        question_examples = question_examples,
                        history = total_result)

  #LLM 응답 파싱 후 저장. (n = LLM이 한 번에 생성하는 질문-SQL쌍 개수 (Prompt에 적혀있는데 따로 수정할 필요는 없는거같아서 냅둠.))
  parsed = parse_llm_output(result, 10)
  total_result.extend(parsed)


#후처리
#질문 말투 다양화
total_result = rephrase_questions(total_result, llm_small, 0.4)

#결과 저장하기.
converted_data = convert_to_gretel_format(total_result, DDL_Statement, DF_NAME, TABLE_NAME)

os.makedirs('data', exist_ok=True)
with open(f'data/Olist_{TABLE_NAME}_text_to_sql_data.json', 'w', encoding='utf-8') as f:
    json.dump(converted_data, f, ensure_ascii=False, indent=4)

print(f"✅ {len(converted_data)}개 변환 완료!")


진행중 : 1
{'input_tokens': 1022, 'output_tokens': 3047, 'total_tokens': 4069, 'input_token_details': {'audio': 0, 'cache_read': 0}, 'output_token_details': {'audio': 0, 'reasoning': 2432}}
진행중 : 2
{'input_tokens': 1207, 'output_tokens': 3907, 'total_tokens': 5114, 'input_token_details': {'audio': 0, 'cache_read': 0}, 'output_token_details': {'audio': 0, 'reasoning': 3200}}
진행중 : 3
{'input_tokens': 1468, 'output_tokens': 2849, 'total_tokens': 4317, 'input_token_details': {'audio': 0, 'cache_read': 0}, 'output_token_details': {'audio': 0, 'reasoning': 2240}}
진행중 : 4
{'input_tokens': 1670, 'output_tokens': 3824, 'total_tokens': 5494, 'input_token_details': {'audio': 0, 'cache_read': 0}, 'output_token_details': {'audio': 0, 'reasoning': 3200}}
진행중 : 5
{'input_tokens': 1955, 'output_tokens': 5865, 'total_tokens': 7820, 'input_token_details': {'audio': 0, 'cache_read': 0}, 'output_token_details': {'audio': 0, 'reasoning': 5056}}
진행중 : 6
{'input_tokens': 2169, 'output_tokens': 5783, 'total_toke

##7.olist_order_customer_dataset

In [192]:
# 변수 설정 (df마다 1,2,3,4 수정해야함.)
DF_NAME = df_customers
TABLE_NAME = "customers"

  #1. DDL 선언문 : INSERT INTO VALUES 삭제
DDL_Statement = """
CREATE TABLE customers (customer_id VARCHAR(32) NOT NULL, customer_unique_id VARCHAR(32) NOT NULL, customer_zip_code_prefix VARCHAR(5) NOT NULL, customer_city VARCHAR(40) NOT NULL, customer_state VARCHAR(2) NOT NULL, PRIMARY KEY (customer_id));
"""

  #2. 칼럼 설명 (README 참고)
column_descriptions = """
customer_id : key to the orders dataset. Each order has a unique customer_id.
customer_unique_id : unique identifier of a customer.
customer_zip_code_prefix : first five digits of customer zip code
customer_city : customer city name
customer_state : customer state
"""

#LLM 실행
total_result = []
for i in range(10):
  print(f'진행중 : {i+1}')

  #3. 칼럼 별 unique한 값 예시
  column_values = get_sample_values(DF_NAME, 3)

  #4. base dataset으로부터 질문 예시 n개 추출
  question_examples = get_sample_queries(base_data, 10)

  result = generate_sql(DDL_Statement = DDL_Statement,
                        column_descriptions = column_descriptions,
                        column_values = column_values,
                        question_examples = question_examples,
                        history = total_result)

  #LLM 응답 파싱 후 저장. (n = LLM이 한 번에 생성하는 질문-SQL쌍 개수 (Prompt에 적혀있는데 따로 수정할 필요는 없는거같아서 냅둠.))
  parsed = parse_llm_output(result, 10)
  total_result.extend(parsed)


#후처리
#질문 말투 다양화
total_result = rephrase_questions(total_result, llm_small, 0.4)

#결과 저장하기.
converted_data = convert_to_gretel_format(total_result, DDL_Statement, DF_NAME, TABLE_NAME)

os.makedirs('data', exist_ok=True)
with open(f'data/Olist_{TABLE_NAME}_text_to_sql_data.json', 'w', encoding='utf-8') as f:
    json.dump(converted_data, f, ensure_ascii=False, indent=4)

print(f"✅ {len(converted_data)}개 변환 완료!")

진행중 : 1
{'input_tokens': 1083, 'output_tokens': 3586, 'total_tokens': 4669, 'input_token_details': {'audio': 0, 'cache_read': 0}, 'output_token_details': {'audio': 0, 'reasoning': 3008}}
진행중 : 2
{'input_tokens': 1258, 'output_tokens': 3411, 'total_tokens': 4669, 'input_token_details': {'audio': 0, 'cache_read': 0}, 'output_token_details': {'audio': 0, 'reasoning': 2688}}
진행중 : 3
{'input_tokens': 1536, 'output_tokens': 4358, 'total_tokens': 5894, 'input_token_details': {'audio': 0, 'cache_read': 0}, 'output_token_details': {'audio': 0, 'reasoning': 3584}}
진행중 : 4
{'input_tokens': 1744, 'output_tokens': 4928, 'total_tokens': 6672, 'input_token_details': {'audio': 0, 'cache_read': 0}, 'output_token_details': {'audio': 0, 'reasoning': 4160}}
진행중 : 5
{'input_tokens': 2019, 'output_tokens': 6898, 'total_tokens': 8917, 'input_token_details': {'audio': 0, 'cache_read': 0}, 'output_token_details': {'audio': 0, 'reasoning': 5952}}
진행중 : 6
{'input_tokens': 2208, 'output_tokens': 5223, 'total_toke

##8.olist_geolocation_dataset

In [194]:
# 변수 설정 (df마다 1,2,3,4 수정해야함.)
DF_NAME = df_geolocation
TABLE_NAME = "geolocation"

  #1. DDL 선언문 : INSERT INTO VALUES 삭제
DDL_Statement = """
CREATE TABLE geolocation (geolocation_zip_code_prefix VARCHAR(5) NOT NULL, geolocation_lat DOUBLE NOT NULL, geolocation_lng DOUBLE NOT NULL, geolocation_city VARCHAR(40) NOT NULL, geolocation_state VARCHAR(2) NOT NULL);
"""

  #2. 칼럼 설명 (README 참고)
column_descriptions = """
geolocation_zip_code_prefix : first 5 digits of zip code
geolocation_lat : latitude
geolocation_lng : longitude
geolocation_city : city name
geolocation_state : state
"""

#LLM 실행
total_result = []
for i in range(10):
  print(f'진행중 : {i+1}')

  #3. 칼럼 별 unique한 값 예시
  column_values = get_sample_values(DF_NAME, 3)

  #4. base dataset으로부터 질문 예시 n개 추출
  question_examples = get_sample_queries(base_data, 10)

  result = generate_sql(DDL_Statement = DDL_Statement,
                        column_descriptions = column_descriptions,
                        column_values = column_values,
                        question_examples = question_examples,
                        history = total_result)

  #LLM 응답 파싱 후 저장. (n = LLM이 한 번에 생성하는 질문-SQL쌍 개수 (Prompt에 적혀있는데 따로 수정할 필요는 없는거같아서 냅둠.))
  parsed = parse_llm_output(result, 10)
  total_result.extend(parsed)


#후처리
#질문 말투 다양화
total_result = rephrase_questions(total_result, llm_small, 0.4)

#결과 저장하기.
converted_data = convert_to_gretel_format(total_result, DDL_Statement, DF_NAME, TABLE_NAME)

os.makedirs('data', exist_ok=True)
with open(f'data/Olist_{TABLE_NAME}_text_to_sql_data.json', 'w', encoding='utf-8') as f:
    json.dump(converted_data, f, ensure_ascii=False, indent=4)

print(f"✅ {len(converted_data)}개 변환 완료!")

진행중 : 1
{'input_tokens': 1017, 'output_tokens': 3398, 'total_tokens': 4415, 'input_token_details': {'audio': 0, 'cache_read': 0}, 'output_token_details': {'audio': 0, 'reasoning': 2560}}
진행중 : 2
{'input_tokens': 1237, 'output_tokens': 4715, 'total_tokens': 5952, 'input_token_details': {'audio': 0, 'cache_read': 0}, 'output_token_details': {'audio': 0, 'reasoning': 3904}}
진행중 : 3
{'input_tokens': 1469, 'output_tokens': 4878, 'total_tokens': 6347, 'input_token_details': {'audio': 0, 'cache_read': 0}, 'output_token_details': {'audio': 0, 'reasoning': 3904}}
진행중 : 4
{'input_tokens': 1717, 'output_tokens': 4359, 'total_tokens': 6076, 'input_token_details': {'audio': 0, 'cache_read': 0}, 'output_token_details': {'audio': 0, 'reasoning': 3392}}
진행중 : 5
{'input_tokens': 2016, 'output_tokens': 4822, 'total_tokens': 6838, 'input_token_details': {'audio': 0, 'cache_read': 0}, 'output_token_details': {'audio': 0, 'reasoning': 3904}}
진행중 : 6
{'input_tokens': 2202, 'output_tokens': 7040, 'total_toke

#2.JOIN 있는 SQL문 생성

## df_orders & df_items

DDL문을 다음과 같이 수정. Human Prompt에 DF으로부터 값 예시 넣는 로직 삭제하고 DDL 문의 VALUES로 대체하기

```
DB에 대한 DDL문을 다음 예시처럼 작성해줘.

#테이블명
orders, order_items

#DB orders
order_id	customer_id	order_status	order_purchase_timestamp	order_approved_at	order_delivered_carrier_date	order_delivered_customer_date	order_estimated_delivery_date
e481f51cbdc54678b7cc49136f2d6af7	9ef432eb6251297304e76186b10a928d	delivered	2017-10-02 10:56:33	2017-10-02 11:07:15	2017-10-04 19:55:00	2017-10-10 21:25:13	2017-10-18 00:00:00
53cdb2fc8bc7dce0b6741e2150273451	b0830fb4747a6c6d20dea0b8c802d7ef	delivered	2018-07-24 20:41:37	2018-07-26 03:24:27	2018-07-26 14:31:00	2018-08-07 15:27:45	2018-08-13 00:00:00
47770eb9100c2d0c44946d9cf07ec65d	41ce2a54c0b03bf3443c3d931a367089	delivered	2018-08-08 08:38:49	2018-08-08 08:55:23	2018-08-08 13:50:00	2018-08-17 18:06:29	2018-09-04 00:00:00

#DB order_items
order_id	order_item_id	product_id	seller_id	shipping_limit_date	price	freight_value
00010242fe8c5a6d1ba2dd792cb16214	1	4244733e06e7ecb4970a6e2683c13e61	48436dade18ac8b2bce089ec2a041202	2017-09-19 09:45:35	58.9	13.29
00018f77f2f0320c557190d7a144bdd3	1	e5f2d52b802189ee658865ca93d83a8f	dd7ddc04e1b6c2c614352b383efe2d36	2017-05-03 11:05:13	239.9	19.93
000229ec398224ef6ca0657da4fc703e	1	c777355d18b72b67abbeef9df44fd0fd	5b51032eddd242adc84c38acab88f23d	2018-01-18 14:48:30	199.0	17.87


#예시
CREATE TABLE salesperson (salesperson_id INT, name TEXT, region TEXT); INSERT INTO salesperson (salesperson_id, name, region) VALUES (1, 'John Doe', 'North'), (2, 'Jane Smith', 'South'); CREATE TABLE timber_sales (sales_id INT, salesperson_id INT, volume REAL, sale_date DATE); INSERT INTO timber_sales (sales_id, salesperson_id, volume, sale_date) VALUES (1, 1, 120, '2021-01-01'), (2, 1, 150, '2021-02-01'), (3, 2, 180, '2021-01-01');

CREATE TABLE farmers_india (id INT, name VARCHAR(255), district_id INT, age INT, income INT); INSERT INTO farmers_india (id, name, district_id, age, income) VALUES (1, 'Farmer A', 1, 45, 50000); CREATE TABLE districts_india (id INT, name VARCHAR(255), state VARCHAR(255)); INSERT INTO districts_india (id, name, state) VALUES (1, 'District A', 'Maharashtra');

CREATE TABLE Armed_Forces (base_id INT, base_name VARCHAR(50), base_location VARCHAR(50), base_type VARCHAR(50)); INSERT INTO Armed_Forces (base_id, base_name, base_location, base_type) VALUES (1, 'Fort Bragg', 'North Carolina', 'Army'); INSERT INTO Armed_Forces (base_id, base_name, base_location, base_type) VALUES (2, 'Camp Pendleton', 'California', 'Marines');
```

In [ ]:
# 변수 설정
  #1. DDL 선언문
  # 굳이 DDL Statement 2개 따로 쓸 필요가 없음

DDL_Statement = "CREATE TABLE orders (order_id VARCHAR(32) NOT NULL, customer_id VARCHAR(32) NOT NULL, order_status VARCHAR(20) NOT NULL, order_purchase_timestamp DATETIME NOT NULL, order_approved_at DATETIME NULL, order_delivered_carrier_date DATETIME NULL, order_delivered_customer_date DATETIME NULL, order_estimated_delivery_date DATETIME NOT NULL, PRIMARY KEY (order_id)); INSERT INTO orders (order_id, customer_id, order_status, order_purchase_timestamp, order_approved_at, order_delivered_carrier_date, order_delivered_customer_date, order_estimated_delivery_date) VALUES ('e481f51cbdc54678b7cc49136f2d6af7', '9ef432eb6251297304e76186b10a928d', 'delivered', '2017-10-02 10:56:33', '2017-10-02 11:07:15', '2017-10-04 19:55:00', '2017-10-10 21:25:13', '2017-10-18 00:00:00'), ('53cdb2fc8bc7dce0b6741e2150273451', 'b0830fb4747a6c6d20dea0b8c802d7ef', 'delivered', '2018-07-24 20:41:37', '2018-07-26 03:24:27', '2018-07-26 14:31:00', '2018-08-07 15:27:45', '2018-08-13 00:00:00'), ('47770eb9100c2d0c44946d9cf07ec65d', '41ce2a54c0b03bf3443c3d931a367089', 'delivered', '2018-08-08 08:38:49', '2018-08-08 08:55:23', '2018-08-08 13:50:00', '2018-08-17 18:06:29', '2018-09-04 00:00:00'); CREATE TABLE order_items (order_id CHAR(32) NOT NULL, order_item_id INT NOT NULL, product_id CHAR(32) NOT NULL, seller_id CHAR(32) NOT NULL, shipping_limit_date DATETIME NOT NULL, price DECIMAL(10,2) NOT NULL, freight_value DECIMAL(10,2) NOT NULL, PRIMARY KEY (order_id, order_item_id), FOREIGN KEY (order_id) REFERENCES orders(order_id)); INSERT INTO order_items (order_id, order_item_id, product_id, seller_id, shipping_limit_date, price, freight_value) VALUES ('00010242fe8c5a6d1ba2dd792cb16214', 1, '4244733e06e7ecb4970a6e2683c13e61', '48436dade18ac8b2bce089ec2a041202', '2017-09-19 09:45:35', 58.90, 13.29), ('00018f77f2f0320c557190d7a144bdd3', 1, 'e5f2d52b802189ee658865ca93d83a8f', 'dd7ddc04e1b6c2c614352b383efe2d36', '2017-05-03 11:05:13', 239.90, 19.93), ('000229ec398224ef6ca0657da4fc703e', 1, 'c777355d18b72b67abbeef9df44fd0fd', '5b51032eddd242adc84c38acab88f23d', '2018-01-18 14:48:30', 199.00, 17.87);"

  #2. 칼럼 설명 (README 참고)
column_descriptions = """
orders
order_id : unique identifier of the order.
customer_id : key to the customer dataset. Each order has a unique customer_id.
order_status : Reference to the order status (delivered, shipped, etc)
order_purchase_timestamp : Shows the purchase timestamp.
order_approved_at : Shows the payment approval timestamp.
order_delivered_carrier_date : Shows the order posting timestamp. When it was handled to the logistic partner.
order_delivered_customer_date : Shows the actual order delivery date to the customer.
order_estimated_delivery_date : Shows the estimated delivery date that was informed to customer at the purchase moment.

order_items
order_id : order unique identifier
order_item_id : sequential number identifying number of items included in the same order.
product_id : product unique identifier
seller_id : seller unique identifier
shipping_limit_date : Shows the seller shipping limit date for handling the order over to the logistic partner.
price : item price
freight_value : item freight value item (if an order has more than one item the freight value is splitted between items)
"""

#LLM 실행
total_result = []
for i in range(1):
  print(f'진행중 : {i+1}')

  #3. 컬럼 값 예시 (unique n개)
  column_values1 = get_sample_values(df_orders, 3)
  column_values2 = get_sample_values(df_order_items, 3)
  column_values = column_values1 + "\n" + column_values2

  #4. 질문 예시
  question_examples = get_sample_queries(base_data, 10)

  result = generate_sql(DDL_Statement = DDL_Statement,
                        column_descriptions = column_descriptions,
                        column_values = column_values,
                        question_examples = question_examples,
                        history = total_result,
                        is_multi=True)


  #LLM 응답 파싱 후 저장.
  parsed = parse_llm_output(result, 3)
  total_result.extend(parsed)

진행중 : 1
{'input_tokens': 1703, 'output_tokens': 290, 'total_tokens': 1993, 'input_token_details': {'audio': 0, 'cache_read': 0}, 'output_token_details': {'audio': 0, 'reasoning': 0}}


In [ ]:
column_values1 = get_sample_values(df_orders, 3)
column_values2 = get_sample_values(df_order_items, 3)

res = column_values1 + "\n" + column_values2
print(res)

order_id : ['18259631aab7299bf25f83f5eac4c8d1', 'f20ad794a21ea34c982e74ec18f7aaec', '73fc8294883c8974e9f856499979579b']
customer_id : ['57d8a458d7250607de1fe62759c5e298', 'bb31f17e002c3cb7358496d89a4e3293', 'dc6a402859b4fb7904e71890a530c63e']
order_status : ['shipped', 'delivered', 'invoiced']
order_purchase_timestamp : ['2017-03-31 12:20:44', '2018-02-04 14:51:57', '2017-05-12 19:05:21']
order_approved_at : ['2018-07-29 18:25:15', '2018-06-17 14:32:13', '2018-08-17 15:29:32']
order_delivered_carrier_date : ['2018-03-20 17:06:53', '2018-02-16 22:39:08', '2017-11-25 13:02:43']
order_delivered_customer_date : ['2017-12-11 22:53:05', '2017-08-04 19:21:53', '2018-06-25 23:18:42']
order_estimated_delivery_date : ['2018-07-24 00:00:00', '2017-09-22 00:00:00', '2018-06-26 00:00:00']

order_id : ['c6948180eccebcf3de3c415cf66c4d31', '7fa29c5dbd1f9eec7b00227f5c389220', '7e7ed851224124724e3058ab01051711']
order_item_id : [14, 18, 20]
product_id : ['1ec486885049bbb9b79351d150ed18c4', '8c90d88939a7

In [ ]:
print(total_result[0].get('SQL'))

SELECT o.order_id, oi.product_id, oi.price, o.order_status
FROM orders o
JOIN order_items oi ON o.order_id = oi.order_id
WHERE DATE(o.order_purchase_timestamp) = '2018-07-24';


In [ ]:
# 파싱된 SQL 전체 DB에 실행해보기.
conn = sqlite3.connect(':memory:') #원래는 sqlite3.connect('mydb.db')로 Disk에 저장. 지금은 :memory:로 RAM에 저장
df_orders.to_sql('orders', conn, if_exists='replace', index=False)
df_order_items.to_sql('order_items', conn, if_exists='replace', index=False)
execute_res = execute_sql_on_db(total_result, conn)

Question:   2017년 12월에 구매된 주문 중 판매자 ID '94165aea8a35c3c21499cbcae239b16c' 또는 '2c54051840f19eca309a5423cf22df36'의 상품이 포함된 주문의 주문 ID, 주문 상태, 주문별 총 상품 가격 합계와 총 배송비 합계를 알려주세요. 총 상품 가격이 큰 순으로 정렬해 주세요.
SQL 원본:   WITH filt_order_ids AS (
  SELECT o.order_id
  FROM orders o
  WHERE o.order_purchase_timestamp >= '2017-12-01' AND o.order_purchase_timestamp < '2018-01-01'
  INTERSECT
  SELECT oi.order_id
  FROM order_items oi
  WHERE oi.seller_id IN ('94165aea8a35c3c21499cbcae239b16c', '2c54051840f19eca309a5423cf22df36')
)
SELECT o.order_id,
       o.order_status,
       SUM(oi.price) AS total_item_price,
       SUM(oi.freight_value) AS total_freight
FROM filt_order_ids f
JOIN orders o ON o.order_id = f.order_id
JOIN order_items oi ON oi.order_id = f.order_id
GROUP BY o.order_id, o.order_status
ORDER BY total_item_price DESC
Status:     success


   affected_rows
0             -1


--------------------------------------------------
Question:   주문 ID '1764b7f40d0e7f04994494ffabf34ecc'에 새 상품 항목을 추가

In [ ]:
#결과 저장하기.
converted_data = convert_to_gretel_format(total_result, DDL_Statement1, DDL_Statement2)

with open('Olist_orders_n_order_items_text_to_sql_data.json', 'w', encoding='utf-8') as f:
    json.dump(converted_data, f, ensure_ascii=False, indent=4)

print(f"✅ {len(converted_data)}개 변환 완료!")

✅ 3개 변환 완료!


In [ ]:
with open('Olist_orders_n_order_items_text_to_sql_data.json', 'r', encoding='utf-8') as f:
    data = json.load(f)
    # print(json.dumps(data, ensure_ascii=False, indent=4))
print(data[0]['instruction'])

DDL statements:

CREATE TABLE orders (
    order_id                      VARCHAR(32)  NOT NULL,          -- 주문 고유 ID (32자리 hex)
    customer_id                   VARCHAR(32)  NOT NULL,          -- 고객 ID (32자리 hex)
    order_status                  VARCHAR(20)  NOT NULL,          -- 주문 상태 (delivered, shipped, canceled 등)
    order_purchase_timestamp      DATETIME     NOT NULL,          -- 주문 생성 시각
    order_approved_at             DATETIME     NULL,              -- 결제 승인 시각 (미승인 시 NULL)
    order_delivered_carrier_date  DATETIME     NULL,              -- 물류사 인계 시각 (배송 전 NULL)
    order_delivered_customer_date DATETIME     NULL,              -- 고객 수령 시각 (미수령 시 NULL)
    order_estimated_delivery_date DATETIME     NOT NULL,          -- 배송 예정일

    PRIMARY KEY (order_id)
);


CREATE TABLE order_items (
    order_id             CHAR(32)      NOT NULL,          -- 주문 ID (orders.order_id 참조)
    order_item_id        INT           NOT NULL,          -- 주문 내 상품 순번
    product_id           CHAR(3

In [ ]:
print(data[0]['output'])

쿼리 작성: WITH filt_order_ids AS (
  SELECT o.order_id
  FROM orders o
  WHERE o.order_purchase_timestamp >= '2017-12-01' AND o.order_purchase_timestamp < '2018-01-01'
  INTERSECT
  SELECT oi.order_id
  FROM order_items oi
  WHERE oi.seller_id IN ('94165aea8a35c3c21499cbcae239b16c', '2c54051840f19eca309a5423cf22df36')
)
SELECT o.order_id,
       o.order_status,
       SUM(oi.price) AS total_item_price,
       SUM(oi.freight_value) AS total_freight
FROM filt_order_ids f
JOIN orders o ON o.order_id = f.order_id
JOIN order_items oi ON oi.order_id = f.order_id
GROUP BY o.order_id, o.order_status
ORDER BY total_item_price DESC;


#3. 데이터 평가/검증



```
# 데이터 형식
1. 모든 list의 원소가 dictionary type인지
2. instruction, input, output 키 존재여부
3. instruction과 output의 값 null 여부

# instruction 값
1. 값이 string 형인지 확인
2. 값에 '입력 텍스트', 'DDL statements' 존재여부
3. '입력 텍스트'가 항상 'DDL statements'보다 앞에 오는지

DDL statements에 INSERT문이 있다면
1. CREATE문에 적힌 테이블명과 INSERT 문에 쓰인 테이블명이 일치하는지
2. CREATE문에 적힌 칼럼명과 INSERT 문에 쓰인 칼럼명이 전체 일치하는지
3. INSERT문에 쓰인 칼럼 개수와 값의 개수가 일치하는지.
4. VALUES 값이 칼럼 데이터형에 맞는 올바른 자료형인지
5. VALUES 값이 칼럼 NULL 허용 여부에 맞는지.
6. 전체 INSERT을 봤을 때 PK의 중복 여부

#input 값
1. 항상 빈 문자열인지 체크

# output 값
1. 값이 string 형인지 확인
2. '쿼리 작성' 존재여부
SQL 확인
1. SQL 실제 실행 되는지 여부
2. SQL이 참조하는 컬럼이 DDL statements에 정의된 칼럼인지 여부

#중복 여부
1. 전체 데이터에서 instruction이 중복되는 것이 있는지 확인
2. 전체 데이터에서 output의 쿼리문이 중복되는 것이 있는지 확인

#3. 전체 데이터에서 instruction의 DDL statement가 중복되는 것이 있는지 확인
```

In [133]:
!pip install sqlglot -q

In [196]:
import importlib
import validate_dataset

importlib.reload(validate_dataset)
from validate_dataset import validate_json_file

In [142]:
#전체 통과
report_orders = validate_json_file("data/Olist_orders_text_to_sql_data.json", llm_sql=llm_sql, _convert_to_sqlite=_convert_to_sqlite)
report_order_items = validate_json_file("data/Olist_order_items_text_to_sql_data.json",llm_sql=llm_sql,_convert_to_sqlite=_convert_to_sqlite)
report_order_payments = validate_json_file("data/Olist_order_payments_text_to_sql_data.json",llm_sql=llm_sql,_convert_to_sqlite=_convert_to_sqlite)
report_products = validate_json_file("data/Olist_products_text_to_sql_data.json",llm_sql=llm_sql,_convert_to_sqlite=_convert_to_sqlite)
report_sellers = validate_json_file("data/Olist_sellers_text_to_sql_data.json",llm_sql=llm_sql,_convert_to_sqlite=_convert_to_sqlite)
report_customers = validate_json_file("data/Olist_customers_text_to_sql_data.json",llm_sql=llm_sql,_convert_to_sqlite=_convert_to_sqlite)


파일: data/Olist_orders_text_to_sql_data.json
총 100건 로드

[데이터 형식 - 원소가 dict 타입인지] 전체 통과
[데이터 형식 - instruction/input/output 키 존재] 전체 통과
[데이터 형식 - instruction 값 not null] 전체 통과
[데이터 형식 - output 값 not null] 전체 통과
[instruction 값 - 값이 string 타입인지] 전체 통과
[instruction 값 - '입력 텍스트' 존재] 전체 통과
[instruction 값 - 'DDL statements' 존재] 전체 통과
[instruction 값 - '입력 텍스트'가 'DDL statements'보다 앞에 위치] 전체 통과
[instruction(INSERT) - CREATE/INSERT 테이블명 일치] 전체 통과
[instruction(INSERT) - CREATE/INSERT 컬럼명 전체 일치] 전체 통과
[instruction(INSERT) - 컬럼 개수와 값 개수 일치] 전체 통과
[instruction(INSERT) - VALUES 값이 컬럼 데이터형에 맞는지] 전체 통과
[instruction(INSERT) - VALUES 값이 NULL 허용 여부에 맞는지] 전체 통과
[instruction(INSERT) - PRIMARY KEY 값 중복 여부] 전체 통과
[input 값 - 항상 빈 문자열인지] 전체 통과
[output 값 - 값이 string 타입인지] 전체 통과
[output 값 - '쿼리 작성' 존재] 전체 통과
[output(SQL) - SQL이 참조하는 컬럼이 DDL에 정의되어 있는지] 전체 통과
[output(SQL) - SQL 실제 실행 가능 여부] 전체 통과
[중복 여부 - instruction 중복] 전체 통과
[중복 여부 - output SQL 중복] 전체 통과


In [154]:
report_order_reviews = validate_json_file("data/Olist_order_reviews_text_to_sql_data.json",llm_sql=llm_sql,_convert_to_sqlite=_convert_to_sqlite)

파일: data/Olist_order_reviews_text_to_sql_data.json
총 100건 로드

[데이터 형식 - 원소가 dict 타입인지] 전체 통과
[데이터 형식 - instruction/input/output 키 존재] 전체 통과
[데이터 형식 - instruction 값 not null] 전체 통과
[데이터 형식 - output 값 not null] 전체 통과
[instruction 값 - 값이 string 타입인지] 전체 통과
[instruction 값 - '입력 텍스트' 존재] 전체 통과
[instruction 값 - 'DDL statements' 존재] 전체 통과
[instruction 값 - '입력 텍스트'가 'DDL statements'보다 앞에 위치] 전체 통과
[instruction(INSERT) - CREATE/INSERT 테이블명 일치] 전체 통과
[instruction(INSERT) - CREATE/INSERT 컬럼명 전체 일치] 전체 통과
[instruction(INSERT) - 컬럼 개수와 값 개수 일치] 전체 통과
[instruction(INSERT) - VALUES 값이 컬럼 데이터형에 맞는지] 전체 통과
[instruction(INSERT) - VALUES 값이 NULL 허용 여부에 맞는지] 전체 통과
[instruction(INSERT) - PRIMARY KEY 값 중복 여부] 전체 통과
[input 값 - 항상 빈 문자열인지] 전체 통과
[output 값 - 값이 string 타입인지] 전체 통과
[output 값 - '쿼리 작성' 존재] 전체 통과
[output(SQL) - SQL이 참조하는 컬럼이 DDL에 정의되어 있는지] 전체 통과
[output(SQL) - SQL 실제 실행 가능 여부] 위반 2건
  - index=75 | 1차 오류=no such function: SUBSTRING_INDEX / LLM 변환 후 오류=near "ORDER": syntax error
  - index=79 | 1차 오류

In [197]:
report_geolocation = validate_json_file("data/Olist_geolocation_text_to_sql_data.json",llm_sql=llm_sql,_convert_to_sqlite=_convert_to_sqlite)

파일: data/Olist_geolocation_text_to_sql_data.json
총 100건 로드

[데이터 형식 - 원소가 dict 타입인지] 전체 통과
[데이터 형식 - instruction/input/output 키 존재] 전체 통과
[데이터 형식 - instruction 값 not null] 전체 통과
[데이터 형식 - output 값 not null] 전체 통과
[instruction 값 - 값이 string 타입인지] 전체 통과
[instruction 값 - '입력 텍스트' 존재] 전체 통과
[instruction 값 - 'DDL statements' 존재] 전체 통과
[instruction 값 - '입력 텍스트'가 'DDL statements'보다 앞에 위치] 전체 통과
[instruction(INSERT) - CREATE/INSERT 테이블명 일치] 전체 통과
[instruction(INSERT) - CREATE/INSERT 컬럼명 전체 일치] 전체 통과
[instruction(INSERT) - 컬럼 개수와 값 개수 일치] 전체 통과
[instruction(INSERT) - VALUES 값이 컬럼 데이터형에 맞는지] 전체 통과
[instruction(INSERT) - VALUES 값이 NULL 허용 여부에 맞는지] 전체 통과
[input 값 - 항상 빈 문자열인지] 전체 통과
[output 값 - 값이 string 타입인지] 전체 통과
[output 값 - '쿼리 작성' 존재] 전체 통과
[output(SQL) - SQL이 참조하는 컬럼이 DDL에 정의되어 있는지] 전체 통과
[output(SQL) - SQL 실제 실행 가능 여부] 위반 2건
  - index=57 | 1차 오류=no such function: STDDEV_POP / LLM 변환 후 오류=no such function: STDEV
  - index=83 | 1차 오류=no such function: STDDEV_SAMP / LLM 변환 후 오류=no such column:

In [198]:
!zip -r data.zip data
print("data.zip created. You can now download it from the files section.")

  adding: data/ (stored 0%)
  adding: data/Olist_geolocation_text_to_sql_data.json (deflated 86%)
  adding: data/Olist_products_text_to_sql_data.json (deflated 87%)
  adding: data/Olist_order_items_text_to_sql_data.json (deflated 79%)
  adding: data/Olist_orders_text_to_sql_data.json (deflated 83%)
  adding: data/Olist_order_reviews_text_to_sql_data.json (deflated 81%)
  adding: data/Olist_order_payments_text_to_sql_data.json (deflated 86%)
  adding: data/Olist_customers_text_to_sql_data.json (deflated 81%)
  adding: data/Olist_sellers_text_to_sql_data.json (deflated 84%)
data.zip created. You can now download it from the files section.
